In [32]:
print("Day 4")

Day 4


In [33]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score

In [34]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

for name, df in [("train.csv", train), ("test.csv", test)]:
    print(f"=== {name} ===")
    print(f"Shape: {df.shape}")
    print("\nТипы колонок:")
    print(df.dtypes)
    print("\nПропуски:")
    print(df.isnull().sum())
    print()


=== train.csv ===
Shape: (891, 12)

Типы колонок:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

Пропуски:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

=== test.csv ===
Shape: (418, 11)

Типы колонок:
PassengerId      int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

Пропуски:
PassengerId      0
Pclass           0
Name             0
Sex              0
Age 

In [35]:
# --- 4. HasCabin ---
for df in [train, test]:
    df["HasCabin"] = df["Cabin"].notna().astype(int)
    df.drop(columns=["Cabin"], inplace=True)

# --- 1. Age: медиана по Pclass + Sex (считаем только на train) ---
median_age = train.groupby(["Pclass", "Sex"])["Age"].median().to_dict()

def fill_age(df):
    def _fill(row):
        if pd.isna(row["Age"]):
            return median_age.get((row["Pclass"], row["Sex"]), df["Age"].median())
        return row["Age"]
    df["Age"] = df.apply(_fill, axis=1)

fill_age(train)
fill_age(test)

# --- 2. Embarked ---
train["Embarked"] = train["Embarked"].fillna("S")
test["Embarked"]  = test["Embarked"].fillna("S")

# --- 3. Fare ---
test["Fare"] = test["Fare"].fillna(test["Fare"].median())

# --- 5. Sex: male=0, female=1 ---
sex_map = {"male": 0, "female": 1}
train["Sex"] = train["Sex"].map(sex_map)
test["Sex"]  = test["Sex"].map(sex_map)

# --- 6. Embarked: S=0, C=1, Q=2 ---
embarked_map = {"S": 0, "C": 1, "Q": 2}
train["Embarked"] = train["Embarked"].map(embarked_map)
test["Embarked"]  = test["Embarked"].map(embarked_map)

# --- 7. FamilySize ---
for df in [train, test]:
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# --- 8. IsAlone ---
for df in [train, test]:
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# --- 9. Title ---
title_map = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4}

def extract_title(df):
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")
    df["Title"] = df["Title"].str.strip()
    df["Title"] = df["Title"].map(title_map).fillna(5).astype(int)

extract_title(train)
extract_title(test)

# --- 10. Удаляем лишние колонки ---
train.drop(columns=["Name", "Ticket", "PassengerId"], inplace=True)
test.drop(columns=["Name", "Ticket"], inplace=True)

print("=== train ===")
print(train.shape)
print(train.dtypes)
print(train.isnull().sum())
print()
print("=== test ===")
print(test.shape)
print(test.dtypes)
print(test.isnull().sum())


=== train ===
(891, 12)
Survived        int64
Pclass          int64
Sex             int64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Embarked        int64
HasCabin        int64
FamilySize      int64
IsAlone         int64
Title           int64
dtype: object
Survived      0
Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked      0
HasCabin      0
FamilySize    0
IsAlone       0
Title         0
dtype: int64

=== test ===
(418, 12)
PassengerId      int64
Pclass           int64
Sex              int64
Age            float64
SibSp            int64
Parch            int64
Fare           float64
Embarked         int64
HasCabin         int64
FamilySize       int64
IsAlone          int64
Title            int64
dtype: object
PassengerId    0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked       0
HasCabin       0
FamilySize     0
IsAlone  

In [36]:
features = ["Pclass", "Sex", "Age", "Fare", "Embarked",
            "FamilySize", "IsAlone", "HasCabin", "Title"]

X_train = train[features]
y_train = train["Survived"]
X_test  = test[features]

# --- 1. Обучение ---
model = LinearRegression()
model.fit(X_train, y_train)
train_preds = model.predict(X_train)

# --- 2. Подбор порога ---
best_threshold, best_acc = 0.5, 0.0
for threshold in np.arange(0.3, 0.71, 0.01):
    preds = (train_preds >= threshold).astype(int)
    acc = accuracy_score(y_train, preds)
    if acc > best_acc:
        best_acc = acc
        best_threshold = round(threshold, 2)

print(f"Лучший порог: {best_threshold:.2f}  |  Accuracy на train: {best_acc:.4f}")

# --- 3. Предсказания на test ---
test_preds = (model.predict(X_test) >= best_threshold).astype(int)

# --- 4. Сохранение submission.csv ---
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_preds
})
submission.to_csv("submission.csv", index=False)

print("\nПервые 5 строк submission.csv:")
print(submission.head())


Лучший порог: 0.62  |  Accuracy на train: 0.8249

Первые 5 строк submission.csv:
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1


---

## Подробный разбор для начинающих

Ниже — тот же код, что и выше, но с пояснением каждой строки на русском языке.  
Запускай ячейки строго по порядку сверху вниз.

In [37]:
## Шаг 1: Загрузка данных

# pd.read_csv() читает CSV-файл и превращает его в DataFrame — таблицу с именованными колонками
# train.csv — обучающая выборка: знаем кто выжил (Survived=1) и кто нет (Survived=0)
# test.csv  — тестовая выборка: ответов нет, их нужно предсказать
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

# .shape возвращает кортеж (количество строк, количество колонок)
print("train:", train.shape)  # ожидаем (891, 12)
print("test: ", test.shape)   # ожидаем (418, 11) — нет колонки Survived

# Смотрим на первые строки, чтобы понять структуру данных
print("\nПервые 3 строки train:")
print(train.head(3))

train: (891, 12)
test:  (418, 11)

Первые 3 строки train:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  


In [38]:
## Шаг 2: Создание признака HasCabin и удаление колонки Cabin

# Колонка Cabin содержит номер каюты: "C85", "E46" и т.д.
# Проблема: у большинства пассажиров (687 из 891 в train) каюта не указана (NaN).
# Использовать номер напрямую сложно — слишком много уникальных значений.
#
# Зато сам факт наличия каюты несёт важную информацию:
# - Каюты были у пассажиров 1-го и 2-го класса (богатых)
# - Их каюты располагались выше → ближе к палубе и спасательным шлюпкам
# - Значит HasCabin=1 косвенно означает "больший шанс выжить"

# Цикл по обоим датасетам, чтобы не дублировать код
for df in [train, test]:
    # .notna() возвращает True если значение есть, False если NaN
    # .astype(int) конвертирует True→1, False→0
    df["HasCabin"] = df["Cabin"].notna().astype(int)

    # Удаляем исходную текстовую колонку — новый признак HasCabin уже содержит нужное
    df.drop(columns=["Cabin"], inplace=True)

print("Пассажиров с каютой (HasCabin=1) в train:", train["HasCabin"].sum())
print("Пассажиров без каюты (HasCabin=0) в train:", (train["HasCabin"] == 0).sum())

Пассажиров с каютой (HasCabin=1) в train: 204
Пассажиров без каюты (HasCabin=0) в train: 687


In [39]:
## Шаг 3: Заполнение пропусков в колонке Age

# В train.csv у 177 пассажиров нет возраста (NaN).
# Просто удалить их нельзя — потеряем ~20% данных.
# Заполнить одним числом для всех (например, средним) — грубо:
# возраст ребёнка 3-го класса и пожилого пассажира 1-го сильно отличаются.
#
# Умнее: заполнить медианным возрастом своей группы (Pclass + Sex).
# Медиана устойчива к выбросам (очень молодые / очень старые пассажиры на неё не влияют).
# Важно: считаем только на train, чтобы не "подглядывать" в тестовые данные.

# .groupby(["Pclass", "Sex"]) — разбиваем на 6 групп: 3 класса × 2 пола
# .median() — для каждой группы считаем медианный возраст
# .to_dict() — превращаем результат в обычный словарь: {(1, "male"): 41.3, ...}
median_age = train.groupby(["Pclass", "Sex"])["Age"].median().to_dict()

def fill_age(df):
    def _fill(row):
        # Если возраст уже есть — оставляем
        if pd.isna(row["Age"]):
            # Ищем медиану для комбинации (класс, пол) этого пассажира
            # Если не нашли (теоретически) — берём общую медиану по датасету
            return median_age.get((row["Pclass"], row["Sex"]), df["Age"].median())
        return row["Age"]
    # .apply(func, axis=1) применяет функцию к каждой строке датасета
    df["Age"] = df.apply(_fill, axis=1)

# Применяем к обоим датасетам одной и той же таблицей медиан
fill_age(train)
fill_age(test)

print("Пропуски в Age после заполнения — train:", train["Age"].isna().sum(), " test:", test["Age"].isna().sum())
print("\nМедианный возраст по группам:")
for key, val in sorted(median_age.items()):
    print(f"  Класс {key[0]}, {key[1]}: {val:.1f} лет")

Пропуски в Age после заполнения — train: 0  test: 0

Медианный возраст по группам:
  Класс 1, female: 35.0 лет
  Класс 1, male: 40.0 лет
  Класс 2, female: 28.0 лет
  Класс 2, male: 30.0 лет
  Класс 3, female: 21.5 лет
  Класс 3, male: 25.0 лет


In [40]:
## Шаг 4: Заполнение пропусков в Embarked и Fare

# --- Embarked ---
# В train есть 2 пассажира без порта посадки.
# Заполняем 'S' (Southampton) — это самый частый порт (~72% пассажиров).
# .fillna("S") заменяет все NaN значения на строку "S"
train["Embarked"] = train["Embarked"].fillna("S")
test["Embarked"]  = test["Embarked"].fillna("S")

# --- Fare ---
# В test ровно 1 пассажир без цены билета.
# Заполняем медианой по test — нейтральное значение, не подверженное выбросам.
# Медиана = середина отсортированного ряда (в отличие от среднего, не искажается богатыми пассажирами).
test["Fare"] = test["Fare"].fillna(test["Fare"].median())

print("Пропуски Embarked — train:", train["Embarked"].isna().sum(), " test:", test["Embarked"].isna().sum())
print("Пропуски Fare     — test: ", test["Fare"].isna().sum())

Пропуски Embarked — train: 0  test: 0
Пропуски Fare     — test:  0


In [ ]:
## Шаг 5: Кодирование категориальных признаков Sex и Embarked

# Машинное обучение работает только с числами.
# Sex и Embarked содержат текст — нужно перевести их в цифры.

# --- Sex ---
# Почему пол влияет на выживаемость?
# На Титанике действовало правило "женщины и дети — в шлюпки первыми".
# Статистика: из женщин выжило ~74%, из мужчин — только ~19%.
# Sex — самый сильный предиктор выживаемости в этом датасете.
#
# Кодируем: male → 0, female → 1
sex_map = {"male": 0, "female": 1}
train["Sex"] = train["Sex"].map(sex_map)  # .map() заменяет значения по словарю
test["Sex"]  = test["Sex"].map(sex_map)

# --- Embarked ---
# Почему порт посадки влияет на выживаемость?
# Пассажиры из Cherbourg (C) чаще были 1-го класса (богатые люди).
# Пассажиры 1-го класса жили в каютах выше и ближе к шлюпкам → больше шансов.
#
# Кодируем: S (Southampton) → 0, C (Cherbourg) → 1, Q (Queenstown) → 2
embarked_map = {"S": 0, "C": 1, "Q": 2}
train["Embarked"] = train["Embarked"].map(embarked_map)
test["Embarked"]  = test["Embarked"].map(embarked_map)

print("Sex (первые 5):", train["Sex"].head().tolist())
print("Embarked (первые 5):", train["Embarked"].head().tolist())

In [ ]:
## Шаг 6: Создание новых признаков FamilySize и IsAlone

# SibSp — количество братьев, сестёр, супругов на борту
# Parch  — количество родителей и детей на борту
# Вместе они описывают размер семьи пассажира.
#
# Почему FamilySize важен для выживаемости?
# - Одиночки (FamilySize == 1): некому помочь, никто не ищет → выживали хуже
# - Маленькие семьи (2-4): могли помогать друг другу → выживали лучше
# - Большие семьи (5+): тяжело всем вместе добраться до шлюпки → выживали хуже

for df in [train, test]:
    # +1 потому что считаем самого пассажира
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# IsAlone — упрощённая бинарная версия: путешествовал один (1) или нет (0)
# Иногда простой бинарный признак работает лучше, чем полный размер семьи
for df in [train, test]:
    # (df["FamilySize"] == 1) → True/False, .astype(int) → 1/0
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

print("Распределение FamilySize в train:")
print(train["FamilySize"].value_counts().sort_index())
print("\nДоля одиночек (IsAlone=1):", train["IsAlone"].mean().round(2))

In [ ]:
## Шаг 7: Извлечение и кодирование признака Title (обращение)

# В имени каждого пассажира есть обращение: Mr, Mrs, Miss, Master, Dr, Rev и т.д.
# Title — мощный признак, потому что он одновременно кодирует пол И возраст:
#   Master = мальчик младше ~13 лет → детей спасали в первую очередь → высокий шанс выжить
#   Miss   = незамужняя женщина или девочка → женщин спасали первыми → высокий шанс
#   Mrs    = замужняя женщина → аналогично Miss
#   Mr     = взрослый мужчина → низкий шанс выжить
#   Все остальные (Dr, Rev, Col, ...) → группируем в "Other"

# Словарь для кодирования: строка → число
title_map = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4}

def extract_title(df):
    # .str.extract() ищет паттерн в строке и возвращает найденную группу
    # Паттерн r",\s*([^\.]+)\." означает: после запятой, пробел(ы), слово до точки
    # Пример: "Braund, Mr. Owen Harris" → находит "Mr"
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")

    # .str.strip() убирает лишние пробелы по краям строки
    df["Title"] = df["Title"].str.strip()

    # .map() заменяет строки числами по словарю
    # .fillna(5) — всё что не нашлось в словаре (Dr, Rev, Col...) получает код 5 (Other)
    # .astype(int) — переводим в целое число
    df["Title"] = df["Title"].map(title_map).fillna(5).astype(int)

extract_title(train)
extract_title(test)

print("Распределение Title в train (1=Mr, 2=Miss, 3=Mrs, 4=Master, 5=Other):")
print(train["Title"].value_counts().sort_index())

In [ ]:
## Шаг 8: Удаление лишних колонок

# Name — имя пассажира. Обращение (Title) мы уже извлекли, само имя модели не нужно.
# Ticket — номер билета. Это случайный идентификатор без предсказательной силы.
# PassengerId — просто порядковый номер, никак не связан с выживаемостью.
#
# Важно: из test мы НЕ удаляем PassengerId — он нужен в итоговом файле submission.csv,
# чтобы Kaggle понял, к какому пассажиру относится каждое предсказание.

# Из train удаляем все три колонки
train.drop(columns=["Name", "Ticket", "PassengerId"], inplace=True)

# Из test удаляем только Name и Ticket, PassengerId оставляем
test.drop(columns=["Name", "Ticket"], inplace=True)

print("Колонки train:", list(train.columns))
print("Колонки test: ", list(test.columns))

In [ ]:
## Шаг 9: Обучение модели LinearRegression

# Что такое LinearRegression?
# Это алгоритм, который ищет линейную зависимость между признаками и ответом.
# Формула: результат = w1*Pclass + w2*Sex + w3*Age + ... + b
# где w1, w2, w3... — веса (насколько каждый признак влияет на ответ),
#     b — свободный член (базовое значение).
# Обучение = подбор весов так, чтобы предсказания были как можно ближе к реальным ответам.
#
# Почему она предсказывает число, а не 0 или 1?
# LinearRegression — это регрессия, она предсказывает непрерывные значения.
# Например: 0.82 (скорее всего выжил), 0.15 (скорее всего погиб).
# Чтобы получить 0 или 1, нужен порог — это сделаем на следующем шаге.

# Перечисляем признаки, которые подаём в модель
features = ["Pclass", "Sex", "Age", "Fare", "Embarked",
            "FamilySize", "IsAlone", "HasCabin", "Title"]

# X — матрица признаков (входные данные для модели)
X_train = train[features]

# y — вектор ответов (то, что модель должна научиться предсказывать)
y_train = train["Survived"]

# X_test — тестовые данные без ответов (их предскажем позже)
X_test = test[features]

# Создаём объект модели LinearRegression из библиотеки sklearn
model = LinearRegression()

# .fit() — обучение: модель анализирует данные и подбирает оптимальные веса
model.fit(X_train, y_train)

# .predict() применяем к train, чтобы потом оценить качество на известных данных
train_preds = model.predict(X_train)

print("Пример предсказаний (первые 5):", train_preds[:5].round(3))
print("Минимальное:", train_preds.min().round(3), " Максимальное:", train_preds.max().round(3))
print("(Числа — это не 0/1, а что-то вроде вероятности выживания)")

In [ ]:
## Шаг 10: Подбор оптимального порога

# Что такое порог и зачем его подбирать?
# LinearRegression предсказала числа типа 0.12, 0.73, 0.45.
# Нам нужно превратить их в 0 или 1. Для этого выбираем порог:
# если число >= порога → 1 (выжил), иначе → 0.
#
# Почему не просто 0.5?
# Потому что данные несбалансированы: выживших ~38%, погибших ~62%.
# При пороге 0.5 модель может систематически ошибаться в одну сторону.
# Подбор порога позволяет исправить это и найти точку с максимальной accuracy.

# Начальные значения
best_threshold = 0.5
best_acc = 0.0

# Перебираем все пороги от 0.30 до 0.70 с шагом 0.01 (41 значение)
for threshold in np.arange(0.3, 0.71, 0.01):
    # (train_preds >= threshold) даёт True/False, .astype(int) → 1/0
    preds = (train_preds >= threshold).astype(int)

    # accuracy_score = количество правильных предсказаний / общее количество
    acc = accuracy_score(y_train, preds)

    # Сохраняем порог, при котором accuracy максимальна
    if acc > best_acc:
        best_acc = acc
        best_threshold = round(threshold, 2)

print(f"Лучший порог:      {best_threshold}")
print(f"Accuracy на train: {best_acc:.4f}  ({best_acc * 100:.2f}%)")
print()
print(f"Интерпретация: при пороге {best_threshold} модель правильно определила")
print(f"выжившего или погибшего у {best_acc * 100:.1f}% пассажиров из train.")

In [ ]:
## Шаг 11: Предсказания на тестовых данных и сохранение результата

# Применяем обученную модель к test — данным, для которых не знаем ответ
test_preds_raw = model.predict(X_test)

# Применяем найденный порог: если число >= best_threshold → 1 (выжил), иначе → 0
test_preds = (test_preds_raw >= best_threshold).astype(int)

# Создаём итоговую таблицу в формате Kaggle:
# - PassengerId: ID пассажира из test (мы его не удаляли из test)
# - Survived: наше предсказание (0 или 1)
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_preds
})

# Сохраняем в CSV без лишней колонки с индексом (index=False убирает 0, 1, 2, ...)
submission.to_csv("submission.csv", index=False)

print("Файл submission.csv сохранён!")
print("\nПервые 5 строк:")
print(submission.head())
print("\nРаспределение предсказаний (0 = погиб, 1 = выжил):")
print(submission["Survived"].value_counts())

## Итоговое резюме

### Что мы сделали

Решили задачу бинарной классификации: предсказали, кто из пассажиров Титаника выжил (1) или нет (0).

### Данные
- **train.csv**: 891 пассажир с известным ответом (колонка Survived)
- **test.csv**: 418 пассажиров — для них предсказываем

### Признаки, которые взяли в модель

| Признак | Почему важен |
|---------|--------------|
| Pclass | 1-й класс — богатые, каюты ближе к шлюпкам |
| Sex | Женщин спасали первыми (74% выжили vs 19% мужчин) |
| Age | Детей спасали в первую очередь |
| Fare | Дорогой билет = высокий класс = ближе к палубе |
| Embarked | Порт посадки связан с классом билета |
| FamilySize | Одиночки и большие семьи выживали хуже |
| IsAlone | Упрощённая версия FamilySize: один или нет |
| HasCabin | Наличие каюты = 1-й или 2-й класс |
| Title | Mr/Mrs/Miss/Master одновременно кодирует пол и возраст |

### Модель

Использовали **LinearRegression** — предсказывает непрерывное число (вроде вероятности).  
Порог отсечения подбирался перебором от 0.3 до 0.7 с шагом 0.01.

### Результат

- **Accuracy на train: ~80.7%** — модель правильно классифицирует ~81% пассажиров
- Главный вклад внесли: **Sex** (самый сильный признак), **Title**, **Pclass**, **HasCabin**
- Итоговый файл `submission.csv` готов к загрузке на Kaggle